# E-Commerce Order Fulfillment Analysis

**Author:** Obadah Elabhari

This notebook provides a reproducible workflow for exploring delivery time, returns, shipping costs, order value, and profit margin.

**Before running:** Place the permitted dataset file named `ecommerce.csv` in the project root folder or update `DATA_PATH` below. The notebook does not include the dataset. The summary figures in the README come from the original project report and should be checked against the actual data.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("../ecommerce.csv")  # Change this path if needed
if not DATA_PATH.exists():
    DATA_PATH = Path("ecommerce.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Could not find ecommerce.csv. Place the permitted dataset in the project root or update DATA_PATH.")

df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
display(df.head())


## 1. Inspect the dataset

In [ ]:
df.info()
display(df.describe(include="all").transpose())


## 2. Data quality checks

In [ ]:
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))
print(f"Duplicate rows: {int(df.duplicated().sum()):,}")


## 3. Summary metrics

In [ ]:
numeric_candidates = ["delivery_time_days", "shipping_cost", "total_amount", "profit_margin", "price", "discount", "quantity"]
available_numeric = [c for c in numeric_candidates if c in df.columns]
display(df[available_numeric].describe().transpose())

if "returned" in df.columns:
    labels = df["returned"].astype(str).str.strip().str.lower()
    returned_mask = labels.isin(["yes", "true", "1", "returned"])
    print(f"Return rate (recognized returned labels): {returned_mask.mean() * 100:.2f}%")
    print("Return labels found:", sorted(df["returned"].dropna().astype(str).unique().tolist()))


## 4. Correlation with profit margin

In [ ]:
cols = [c for c in ["delivery_time_days", "shipping_cost", "total_amount", "profit_margin", "price", "discount", "quantity", "customer_age"] if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
if "profit_margin" in cols and len(cols) > 1:
    corr = df[cols].corr(numeric_only=True)["profit_margin"].drop("profit_margin")
    display(corr.sort_values(ascending=False).to_frame("correlation_with_profit_margin"))
    ax = corr.sort_values().plot(kind="bar", figsize=(9, 4))
    ax.set_title("Correlation with Profit Margin")
    ax.set_ylabel("Correlation coefficient")
    ax.set_xlabel("Variable")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Required numeric columns were not found; review the dataset column names.")


## 5. Average delivery time by region

In [ ]:
if {"region", "delivery_time_days"}.issubset(df.columns):
    regional = df.groupby("region", dropna=False)["delivery_time_days"].mean().sort_values(ascending=False)
    display(regional.to_frame("average_delivery_time_days"))
    ax = regional.sort_values().plot(kind="bar", figsize=(8, 4))
    ax.set_title("Average Delivery Time by Region")
    ax.set_ylabel("Average delivery time (days)")
    ax.set_xlabel("Region")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Columns 'region' and/or 'delivery_time_days' were not found.")


## 6. Return-rate visualization

In [ ]:
if "returned" in df.columns:
    counts = df["returned"].astype(str).value_counts(dropna=False)
    ax = counts.plot(kind="bar", figsize=(7, 4))
    ax.set_title("Orders by Return Status")
    ax.set_ylabel("Number of orders")
    ax.set_xlabel("Return status")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Column 'returned' was not found.")


## 7. Interpretation and next steps

Use the actual notebook outputs to write your final interpretation. Avoid claiming that correlation proves causation. Investigate differences by region, product category, and return status, and validate unusual values before making business recommendations.

### Portfolio checklist
- [ ] Confirm the dataset license and cite its original source.
- [ ] Run all cells successfully with the real CSV.
- [ ] Check that results match the report.
- [ ] Export charts into the `images/` folder if desired.
- [ ] Remove any private or identifying information before publishing.
